# PRS

In [ ]:
cd "/path/to/analysis/PRS"

In [ ]:
IMPUTED="/path/to/imputed/files"

## Get profile files

### Extract Nalls et al. 2019 90 risk loci

Extract all risk loci from each chromosome and merge all together.

In [ ]:
seq 1 22 | parallel -j 12 \
  plink2 --vcf ${IMPUTED}/chr{}.dose.vcf.gz \
    --const-fid \
    --set-all-var-ids 'chr@:#:\$r:\$a' \
    --new-id-max-allele-len 10000 missing \
    --threads 1 \
    --maf 0.01 \
    --hwe 1e-6 \
    --geno 0.05 \
    --mind 0.05 \
    --extract range ./SCORE/Nalls_2019_EUR.range \
    --make-pgen \
    --out ./INPUT/chr{}_Nalls_2019_90LOCI

In [ ]:
# Merge all chromosomes
ls ./INPUT/chr*_Nalls_2019_90LOCI.pgen | sed 's/.pgen//' > ./INPUT/Nalls_2019_90LOCI.merge
cat ./INPUT/Nalls_2019_90LOCI.merge

In [ ]:
# Merge files
plink2 --pmerge-list ./INPUT/Nalls_2019_90LOCI.merge \
  --sort-vars \
  --make-pgen \
  --out ./INPUT/CATPD_Nalls_2019_90LOCI

In [ ]:
sed 's/^0_//' ./INPUT/CATPD_Nalls_2019_90LOCI.psam > temp
rm ./INPUT/CATPD_Nalls_2019_90LOCI.psam
cp temp ./INPUT/CATPD_Nalls_2019_90LOCI.psam
head ./INPUT/CATPD_Nalls_2019_90LOCI.psam

In [ ]:
# Merge files
plink2 --pfile ./INPUT/CATPD_Nalls_2019_90LOCI\
  --keep ./INPUT/CATPD_unrelated.keep \
  --make-bed \
  --out ./INPUT/CATPD_Nalls_2019_90LOCI

In [ ]:
# Get profile file
plink --bfile ./INPUT/CATPD_Nalls_2019_90LOCI \
  --score ./SCORE/Nalls_2019_EUR.txt 1 2 3 header \
  --out ./OUTPUT/CATPD_Nalls_2019_90LOCI

In [ ]:
cat ./OUTPUT/CATPD_Nalls_2019_90LOCI.nopred

### Extract Leonard et al. 2025 156 risk loci

Extract all risk loci from each chromosome and merge all together.

In [ ]:
seq 1 22 | parallel -j 12 \
  plink2 --vcf ${IMPUTED}/chr{}.dose.vcf.gz \
    --const-fid \
    --set-all-var-ids 'chr@:#:\$r:\$a' \
    --new-id-max-allele-len 10000 missing \
    --threads 1 \
    --maf 0.01 \
    --hwe 1e-6 \
    --geno 0.05 \
    --mind 0.05 \
    --extract range ./SCORE/Leonard_2025_EUR.range \
    --make-pgen \
    --out ./INPUT/chr{}_Leonard_2025_156LOCI

In [ ]:
# Merge all chromosomes
ls ./INPUT/chr*_Leonard_2025_156LOCI.pgen | sed 's/.pgen//' > ./INPUT/Leonard_2025_156LOCI.merge
cat ./INPUT/Leonard_2025_156LOCI.merge

In [ ]:
# Merge files
plink2 --pmerge-list ./INPUT/Leonard_2025_156LOCI.merge \
  --sort-vars \
  --make-pgen \
  --out ./INPUT/CATPD_Leonard_2025_156LOCI

In [ ]:
sed 's/^0_//' ./INPUT/CATPD_Leonard_2025_156LOCI.psam > temp
rm ./INPUT/CATPD_Leonard_2025_156LOCI.psam
cp temp ./INPUT/CATPD_Leonard_2025_156LOCI.psam
head ./INPUT/CATPD_Leonard_2025_156LOCI.psam

In [ ]:
# Merge files
plink2 --pfile ./INPUT/CATPD_Leonard_2025_156LOCI\
  --make-bed \
  --keep ./INPUT/CATPD_unrelated.keep \
  --out ./INPUT/CATPD_Leonard_2025_156LOCI

In [ ]:
# Get profile file
plink --bfile ./INPUT/CATPD_Leonard_2025_156LOCI \
  --score ./SCORE/Leonard_2025_EUR.txt 1 2 3 header \
  --out ./OUTPUT/CATPD_Leonard_2025_156LOCI

## Change kernel to R

Make sure ID column in covariate file is "IID"

In [ ]:
library(tidyverse)
library(patchwork)
library(fmsb)
library(pROC)
library(knitr)

In [ ]:
setwd('/path/to/analysis/DATA/ANALYSIS/PRS')

### Load Nalls PRS

In [ ]:
# Load Nalls 90 SNP scores
scores_nalls <- read_table("./OUTPUT/CATPD_Nalls_2019_90LOCI.profile")
pheno  <- read_table("INPUT/CATPD_covariate.cov") 

# Merge with phenotype
df_nalls <- inner_join(
  scores_nalls %>% select(IID, SCORE),
  pheno,
  by = "IID"
) %>% filter(!is.na(DISEASE), !is.na(AGE_AAO), !is.na(SEX))

df_nalls$SCORE_Z <- as.numeric(scale(df_nalls$SCORE))

# Run model_leonard
model_nalls <- glm(DISEASE ~ scale(SCORE) + SEX + AGE_AAO +
                     GCTA_PC1 + GCTA_PC2 + GCTA_PC3 + GCTA_PC4 + GCTA_PC5 +
                     GCTA_PC6 + GCTA_PC7 + GCTA_PC8 + GCTA_PC9 + GCTA_PC10,
                   data = df_nalls, family = binomial)

summary(model_nalls)

### Load Leonard PRS

In [ ]:
# Load Nalls 90 SNP scores
scores_leonard <- read_table("./OUTPUT/CATPD_Leonard_2025_156LOCI.profile")
pheno  <- read_table("INPUT/CATPD_covariate.cov") 

# Merge with phenotype
df_leonard <- inner_join(
  scores_leonard %>% select(IID, SCORE),
  pheno,
  by = "IID"
) %>% filter(!is.na(DISEASE), !is.na(AGE_AAO), !is.na(SEX))

df_leonard$SCORE_Z <- as.numeric(scale(df_leonard$SCORE))

# Run model_leonard
model_leonard <- glm(DISEASE ~ scale(SCORE) + SEX + AGE_AAO +
                     GCTA_PC1 + GCTA_PC2 + GCTA_PC3 + GCTA_PC4 + GCTA_PC5 +
                     GCTA_PC6 + GCTA_PC7 + GCTA_PC8 + GCTA_PC9 + GCTA_PC10,
                   data = df_leonard, family = binomial)

summary(model_leonard)

### Compare

In [ ]:
# Leonard 2025
roc_leonard <- suppressMessages(roc(df_leonard$DISEASE, fitted(model_leonard)))
r2_leonard  <- NagelkerkeR2(model_leonard)$R2

# Nalls 2019
roc_nalls   <- suppressMessages(roc(df_nalls$DISEASE, fitted(model_nalls)))
r2_nalls    <- NagelkerkeR2(model_nalls)$R2

# Null model_leonard (covariates only, use either dataset - should be same)
model_null <- glm(DISEASE ~ SEX + AGE_AAO +
                    GCTA_PC1 + GCTA_PC2 + GCTA_PC3 + GCTA_PC4 + GCTA_PC5 +
                    GCTA_PC6 + GCTA_PC7 + GCTA_PC8 + GCTA_PC9 + GCTA_PC10,
                  data = df_leonard, family = binomial)
r2_null  <- NagelkerkeR2(model_null)$R2
auc_null <- suppressMessages(auc(roc(df_leonard$DISEASE, fitted(model_null))))

In [ ]:
tibble(
  model_leonard         = c("Null (covariates)", "Nalls 2019 (90 SNPs)", "Leonard 2025 (156 SNPs)"),
  AUC           = round(c(auc_null, auc(roc_nalls), auc(roc_leonard)), 3),
  Nagelkerke_R2 = round(c(r2_null, r2_nalls, r2_leonard), 4),
  Delta_R2      = round(c(NA, r2_nalls - r2_null, r2_leonard - r2_null), 4)
)

In [ ]:
library(fmsb)
library(pROC)

# Null model_leonard
model_null <- glm(DISEASE ~ SEX + AGE_AAO +
                    GCTA_PC1 + GCTA_PC2 + GCTA_PC3 + GCTA_PC4 + GCTA_PC5 +
                    GCTA_PC6 + GCTA_PC7 + GCTA_PC8 + GCTA_PC9 + GCTA_PC10,
                  data = df_leonard, family = binomial)

# ROC objects
roc_null    <- suppressMessages(roc(df_leonard$DISEASE, fitted(model_null)))
roc_leonard <- suppressMessages(roc(df_leonard$DISEASE, fitted(model_leonard)))
roc_nalls   <- suppressMessages(roc(df_nalls$DISEASE,   fitted(model_nalls)))

# R2
r2_null    <- NagelkerkeR2(model_null)$R2
r2_leonard <- NagelkerkeR2(model_leonard)$R2
r2_nalls   <- NagelkerkeR2(model_nalls)$R2

# OR and CI
or_leonard <- broom::tidy(model_leonard, conf.int=TRUE, exponentiate=TRUE) %>%
  filter(term == "scale(SCORE)")
or_nalls <- broom::tidy(model_nalls, conf.int=TRUE, exponentiate=TRUE) %>%
  filter(term == "scale(SCORE)")

# Build table
perf_table <- tibble(
  model_leonard          = c("Null (covariates only)",
                     "Nalls 2019 (90 SNPs)",
                     "Leonard 2025 (156 SNPs)"),
  N_samples      = c(nrow(df_leonard), nrow(df_nalls), nrow(df_leonard)),
  OR_per_SD      = c(NA,
                     round(or_nalls$estimate, 3),
                     round(or_leonard$estimate, 3)),
  CI_95          = c(NA,
                     paste0(round(or_nalls$conf.low, 2), "-", round(or_nalls$conf.high, 2)),
                     paste0(round(or_leonard$conf.low, 2), "-", round(or_leonard$conf.high, 2))),
  P_value        = c(NA,
                     formatC(or_nalls$p.value, format="e", digits=2),
                     formatC(or_leonard$p.value, format="e", digits=2)),
  AUC            = round(c(auc(roc_null), auc(roc_nalls), auc(roc_leonard)), 3),
  Nagelkerke_R2  = round(c(r2_null, r2_nalls, r2_leonard), 4),
  Delta_R2       = round(c(NA, r2_nalls - r2_null, r2_leonard - r2_null), 4)
)

# kable(perf_table, align = "lcccccccc")
perf_table

In [ ]:
write_csv(perf_table, "./OUTPUT/prs_performance_table.csv")

### Plot

In [ ]:
dist_df <- bind_rows(
  df_leonard %>% select(SCORE_Z, DISEASE) %>% mutate(PRS = "Leonard 2025"),
  df_nalls    %>% select(SCORE_Z, DISEASE) %>% mutate(PRS = "Nalls 2019")
)

or_df <- bind_rows(
  broom::tidy(model_leonard, conf.int=TRUE, exponentiate=TRUE) %>%
    filter(term == "scale(SCORE)") %>% mutate(PRS = "Leonard 2025\n(156 SNPs)"),
  broom::tidy(model_nalls, conf.int=TRUE, exponentiate=TRUE) %>%
    filter(term == "scale(SCORE)") %>% mutate(PRS = "Nalls 2019\n(90 SNPs)")
)

r2_df <- tibble(
  PRS     = c("Nalls 2019\n(90 SNPs)", "Leonard 2025\n(156 SNPs)"),
  DeltaR2 = c(r2_nalls - r2_null, r2_leonard - r2_null)
)

# ── 2. ROC curves overlaid ──────────────────────────────────────
roc_leonard <- suppressMessages(roc(df_leonard$DISEASE, fitted(model_leonard)))
roc_nalls   <- suppressMessages(roc(df_nalls$DISEASE,    fitted(model_nalls)))
roc_null    <- suppressMessages(roc(df_leonard$DISEASE, fitted(model_null)))

roc_df <- bind_rows(
  data.frame(
    specificity = rev(roc_leonard$specificities),
    sensitivity = rev(roc_leonard$sensitivities),
    model_leonard = paste0("Leonard 2025 (AUC=", round(auc(roc_leonard), 3), ")")
  ),
  data.frame(
    specificity = rev(roc_nalls$specificities),
    sensitivity = rev(roc_nalls$sensitivities),
    model_leonard = paste0("Nalls 2019 (AUC=", round(auc(roc_nalls), 3), ")")
  ),
  data.frame(
    specificity = rev(roc_null$specificities),
    sensitivity = rev(roc_null$sensitivities),
    model_leonard = paste0("Null (AUC=", round(auc(roc_null), 3), ")")
  )
)

In [ ]:
# Define colors
col_case    <- "#5BBCD6"  # teal
col_control <- "#B09FCA"  # light violet

# Consistent theme
base_theme <- theme_classic(base_size = 16) +
  theme(
    plot.title   = element_text(size = 17, face = "bold"),
    axis.title   = element_text(size = 16),
    axis.text    = element_text(size = 15),
    legend.text  = element_text(size = 15),
    legend.title = element_text(size = 15),
    strip.text   = element_text(size = 16),
    strip.background = element_blank(),
    plot.margin  = margin(10, 15, 10, 15)
  )

p1 <- ggplot(dist_df, aes(x = SCORE_Z, fill = factor(DISEASE))) +
  geom_density(alpha = 0.5) +
  scale_fill_manual(values = c(col_control, col_case),
                    labels = c("Control", "Case")) +
  facet_wrap(~PRS) +
  labs(title = "PRS Distribution", x = "PRS (Z-score)",
       y = "Density", fill = NULL) +
  base_theme

p2 <- ggplot(roc_df, aes(x = 1 - specificity, y = sensitivity, color = model_leonard)) +
  geom_line(linewidth = 0.6) +
  geom_abline(linetype = "dashed", color = "grey50", linewidth = 0.4) +
  scale_color_manual(values = c(col_case, col_control, "grey60")) +
  labs(title = "ROC Curves", x = "1 - Specificity",
       y = "Sensitivity", color = NULL) +
  base_theme +
  theme(legend.position = c(0.62, 0.15))

p3 <- ggplot(or_df, aes(x = PRS, y = estimate,
                         ymin = conf.low, ymax = conf.high, color = PRS)) +
  geom_pointrange(size = 0.4, linewidth = 0.6) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "grey50", linewidth = 0.4) +
  scale_color_manual(values = c(col_case, col_control)) +
  labs(title = "OR per SD", x = NULL, y = "Odds Ratio") +
  base_theme +
  theme(legend.position = "none")

p4 <- ggplot(r2_df, aes(x = PRS, y = DeltaR2, fill = PRS)) +
  geom_col(width = 0.4) +
  geom_text(aes(label = round(DeltaR2, 4)), vjust = -0.5, size = 5) +
  scale_fill_manual(values = c(col_case, col_control)) +
  labs(title = "PRS Contribution (ΔNagelkerke R²)",
       x = NULL, y = "ΔR²") +
  base_theme +
  theme(legend.position = "none")

combined <- (p1) / (p2 | p3 | p4) +
  plot_layout(heights = c(1, 1.2))

ggsave("./OUTPUT/prs_comparison.png", combined,
       width = 20, height = 14, dpi = 300)
combined